## 练习1

In [198]:
import pandas as pd
import numpy as np

raw_data = {
    'record_id': [1001, 1002, 1003, 1004, 1005, 1006],
    'age': [25, -3, 28.2, 150, 32, np.nan], 
    'join_date': ['2023-01-15', '2023/02/28', '15-03-2023', 'Not a Date', '2023-05-10', '2023-06-01'], 
    'score': ['88.5', 92.0, np.nan, 'Error_Code_X', 75.5, 80.0] 
}

df = pd.DataFrame(raw_data)

In [199]:
# 查看数据信息
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   record_id  6 non-null      int64  
 1   age        5 non-null      float64
 2   join_date  6 non-null      object 
 3   score      5 non-null      object 
dtypes: float64(1), int64(1), object(2)
memory usage: 324.0+ bytes
None


In [200]:
# record_id列类型显示为int64,ID没有实际的数学意义，转为str
data_clean = df.copy()
data_clean['record_id'] = data_clean['record_id'].astype(str)
print(data_clean['record_id'].dtype)

object


In [201]:
# 年龄应该是整数类型
# 查看年龄列的数值范围
print(data_clean['age'].describe())

# 去除age列的异常值
# 抓取无效年龄值（布尔序列）
invalid_age_mask = (data_clean['age'] < 0) | (data_clean['age'] > 120)
# print(invalid_age_mask)
# print(data_clean[invalid_age_mask])
# 将无效年龄替换为NaN
data_clean.loc[invalid_age_mask,'age'] = np.nan
# print(data_clean)

# 将age列的浮点数向下取整,并转换为整数类型
data_clean['age'] = np.floor(data_clean['age']).astype('Int8')
print(f"查看清洗后的age列\n{data_clean['age']}")
print(data_clean['age'].dtype)



count      5.000000
mean      46.440000
std       59.518636
min       -3.000000
25%       25.000000
50%       28.200000
75%       32.000000
max      150.000000
Name: age, dtype: float64
查看清洗后的age列
0      25
1    <NA>
2      28
3    <NA>
4      32
5    <NA>
Name: age, dtype: Int8
Int8


In [202]:
# 处理join_date

# 由于不知道join_data的脏数据类型，因此采取“压力测试”
parsed_dates = pd.to_datetime(data_clean['join_date'],errors='coerce',format='mixed')
print(parsed_dates)

# 找出原本不是空值，被解析后变为空值的数据（即脏数据类型）
mask_bad_dates = (data_clean['join_date'].notna()) & (parsed_dates.isna())

bad_date_examples = data_clean.loc[mask_bad_dates,'join_date']
print(bad_date_examples)

# 如果脏数据类型数量过大，不可能全部打印出来。
# 不需要看清每一个长什么样，只需要看哪种脏类型出现得最频繁
dirty_stype_rank = bad_date_examples.value_counts().head()
print(f"脏数据排行榜：")
print(dirty_stype_rank)

# 如果脏数据类型非常分散，几万条脏数据类型都不一样，就要把具体得文本抽象化，统计脏数据得长度分布
# 正常日期长度一般为10
length_distribution = bad_date_examples.str.len().value_counts()
print(length_distribution)

# 将data_clean得join_date替换为parsed_dates
data_clean['join_date'] = parsed_dates
print(data_clean['join_date'])

0   2023-01-15
1   2023-02-28
2   2023-03-15
3          NaT
4   2023-05-10
5   2023-06-01
Name: join_date, dtype: datetime64[ns]
3    Not a Date
Name: join_date, dtype: object
脏数据排行榜：
join_date
Not a Date    1
Name: count, dtype: int64
join_date
10    1
Name: count, dtype: int64
0   2023-01-15
1   2023-02-28
2   2023-03-15
3          NaT
4   2023-05-10
5   2023-06-01
Name: join_date, dtype: datetime64[ns]


In [203]:
# 清洗score

# 上大重量做压力测试，将score强转为数值，遇到无法解析得乱码，强制转为NaN
parsed_scores = pd.to_numeric(data_clean['score'],errors='coerce')
print(parsed_scores)

# 找出转换前不为空值，但转换后为空值的数据
mask_bad_score = (data_clean['score'].notna()) & (parsed_scores.isna())
bad_score_examples = data_clean.loc[mask_bad_score,'score']
print(bad_score_examples)

# 统计脏数据类型数量排行
score_dirty_stype_rank = bad_score_examples.value_counts().head()
print(score_dirty_stype_rank)

# 将清洗后的数据存进data_clean
data_clean['score'] = parsed_scores
print(data_clean)
print(data_clean.info())

0    88.5
1    92.0
2     NaN
3     NaN
4    75.5
5    80.0
Name: score, dtype: float64
3    Error_Code_X
Name: score, dtype: object
score
Error_Code_X    1
Name: count, dtype: int64
  record_id   age  join_date  score
0      1001    25 2023-01-15   88.5
1      1002  <NA> 2023-02-28   92.0
2      1003    28 2023-03-15    NaN
3      1004  <NA>        NaT    NaN
4      1005    32 2023-05-10   75.5
5      1006  <NA> 2023-06-01   80.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   record_id  6 non-null      object        
 1   age        3 non-null      Int8          
 2   join_date  5 non-null      datetime64[ns]
 3   score      4 non-null      float64       
dtypes: Int8(1), datetime64[ns](1), float64(1), object(1)
memory usage: 288.0+ bytes
None


## 练习2

In [204]:
import pandas as pd
import numpy as np

# --- 靶场数据：星际克隆人服役档案 (Clone Registry) ---
raw_data = {
    'clone_id': [' C-137 ', 'c_138', 'C-139\n', 'X-999', ' c-140'],
    'comm_link': ['+86 138-1234-5678', '001-(800)-555-0199', 'Disconnected', '13911112222_ext', '137-0000-1111'],
    'combat_class': [' assault ', 'SNIPER', 'Medic', 'Berserker', 'assault'],
    'age': [16, 25, 38, 150, 42],
    'vitals_score': ['98.5', 'Error_Timeout', '105.0', 88.5, np.nan]
}
df = pd.DataFrame(raw_data)
print("🚨 原始被污染的克隆人档案：")
print(df_clones)

🚨 原始被污染的克隆人档案：
  clone_id           comm_link combat_class  age   vitals_score
0   C-137    +86 138-1234-5678     assault    16           98.5
1    c_138  001-(800)-555-0199       SNIPER   25  Error_Timeout
2  C-139\n        Disconnected        Medic   38          105.0
3    X-999     13911112222_ext    Berserker  150           88.5
4    c-140       137-0000-1111      assault   42            NaN


In [205]:
# 数据概览
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   clone_id      5 non-null      object
 1   comm_link     5 non-null      object
 2   combat_class  5 non-null      object
 3   age           5 non-null      int64 
 4   vitals_score  4 non-null      object
dtypes: int64(1), object(4)
memory usage: 332.0+ bytes


In [206]:
# 备份数据
df_clean = df.copy()

In [207]:
# 处理combat_class

# 创建合法兵种名单
valid_classes = {'assault', 'sniper', 'medic'}

# 将combat_class去空格并转小写
df_clean['combat_class'] = df_clean['combat_class'].str.strip().str.lower()

# 筛选出不合法的名字
inconsistent_classes = set(df_clean['combat_class']).difference(set(valid_classes))
print(inconsistent_classes)

# 生成combat_class列数据是否为不合法类别的布尔序列
is_dirty = df_clean['combat_class'].isin(inconsistent_classes)
print(is_dirty)

# 将combat_class列的不合法类别转为NaN
df_clean.loc[is_dirty,'combat_class']=np.nan
print(df_clean)


{'berserker'}
0    False
1    False
2    False
3     True
4    False
Name: combat_class, dtype: bool
  clone_id           comm_link combat_class  age   vitals_score
0   C-137    +86 138-1234-5678      assault   16           98.5
1    c_138  001-(800)-555-0199       sniper   25  Error_Timeout
2  C-139\n        Disconnected        medic   38          105.0
3    X-999     13911112222_ext          NaN  150           88.5
4    c-140       137-0000-1111      assault   42            NaN


In [ ]:
# 1. 暴力推平：剥离所有非数字字符
df_clean['comm_link'] = df_clean['comm_link'].str.replace(r'\D', '', regex=True)

# 2. 格式统一：把 13 位的号码切掉前两位，强制对齐
df_clean['comm_link'] = np.where(
    df_clean['comm_link'].str.len() == 13,
    df_clean['comm_link'].str[2:],
    df_clean['comm_link']
)

# 3. 终局宣判：只挥这一刀！所有长度小于 10 的（包括0、5、9等），全部转为 NaN
df_clean.loc[df_clean['comm_link'].str.len() < 10, 'comm_link'] = np.nan

# 4. 黄金验收：死死锁住底层逻辑
assert (df_clean['comm_link'].str.contains(r'[+|-|\(\)]').any() == False)
print("✅ comm_link 清洗彻底完成！")

0    8613812345678
1    0018005550199
2                 
3      13911112222
4      13700001111
Name: comm_link, dtype: object
0    8613812345678
1    0018005550199
2              NaN
3      13911112222
4      13700001111
Name: comm_link, dtype: object
0    13812345678
1    18005550199
2            NaN
3    13911112222
4    13700001111
Name: comm_link, dtype: object


### 真实业务场景关于Phone Number的清洗

* **在真实的工业界，如果我们真敢把 `df_clean['comm_link'].str.len() < 10` 这把刀劈下去，明天早上业务线负责人就会拿着刀来找我们：**

    - 市话座机号（无区号）：比如用户只填了 87654321（8位）或者 1234567（7位）。

    - 特服号码：银行客服 95588（5位），甚至紧急电话 110（3位）。

    - 带分机号的异形座机：比如刚才靶场里的 13911112222_ext（真实场景下可能是 010-12345678-801，如果你把非数字全删了，区号、主号、分机号全粘在一起，根本无法拨打）。

* **所以，面对真实的、全球化的、甚至带有本地缩写的电话号码清洗，硬写正则表达式和长度判断，是一条注定会走向死胡同的绝路。**

* **🌟 工业界的终极杀器：把专业的事交给专业的库**

    在真实的大厂数据清洗中，碰到“手机号/座机号”，我们绝对不会自己去发明车轮。全世界的电话号码规则太变态了，只有 Google 维护的底层开源库才能镇得住场子。

    在 Python 里，这个神级兵器叫 phonenumbers。

* **如果是在真正的生产环境里，我们根本不写什么 \D 和 str.len()，我们会直接呼叫这个库进行“降维打击”。它的脑回路是这样的：**

```python
import pandas as pd
import numpy as np
# 真实生产环境中，需要 pip install phonenumbers
import phonenumbers 

# 定义一个极度聪明的微创手术刀函数
def clean_real_phone(phone_str):
    if pd.isna(phone_str):
        return np.nan
        
    try:
        # 1. 把它丢给底层解析器（假设我们这批克隆人的默认基站是地球/中国区 'CN'）
        # 它懂得识别 +86，懂得区分座机、手机、甚至是带分机号的格式
        parsed_num = phonenumbers.parse(str(phone_str), "CN")
        
        # 2. 调用全球号码规则库进行“物理验证”
        if phonenumbers.is_valid_number(parsed_num):
            # 3. 如果合法，强制输出为国际标准 E.164 格式 (比如 +8613812345678)
            return phonenumbers.format_number(parsed_num, phonenumbers.PhoneNumberFormat.E164)
        else:
            # 不合法的（比如那堆字母乱码），直接枪毙
            return np.nan
    except:
        return np.nan

# 一键应用到整列
# df_clean['comm_link'] = df_clean['comm_link'].apply(clean_real_phone)
```

* **代码解读：**
    * **🛠️ phonenumbers 库核心三大项**
        在 Jupyter 里，需要先安装它（pip install phonenumbers）。它的核心工作流永远是这严格的三步：`解析 (Parse)` -> `校验 (Validate)` -> `格式化 (Format)`。

        * **第一步：解析器 (Parser) —— 赋予字符串物理意义**

            - 机器拿到的只是文本，我们需要强制让它以“电话号码”的结构去理解这段文本。

            ```python
            import phonenumbers

            # 模拟一个脏乱差的输入，带有空格和无意义的符号
            raw_string = "  010- 1234 5678  "

            # parse 的第二个参数 "CN" 是极其关键的“默认防线（Default Region）”
            # 如果号码自带国际区号（比如 +86），它会优先认 +86；
            # 如果没有区号（像上面这个），它就会默认把它当做中国大陆（CN）的号码来解析。
            parsed_obj = phonenumbers.parse(raw_string, "CN")

            print(parsed_obj)
            # 输出: Country Code: 86 National Number: 1012345678
            ```
            `底层逻辑：` 这一步它并没有判断号码能不能打通，它只是把文本强行拆解成了“国家码 + 国内号码”的对象结构。
        
        * **第二步：校验器 (Validator) —— 呼叫全球规则库**

            - 这是这个库最值钱的一步。它会拿着解析出来的对象，去对比底层的规则网络。

            ```python
            # 验证这个号码是否符合该国家的真实号段和长度规则
            is_valid = phonenumbers.is_valid_number(parsed_obj)

            print(is_valid) 
            # 输出: True (因为 010 确实是合法的北京区号，且后续长度合法)

            # 如果你传入的是 phonenumbers.parse("12345", "CN")
            # is_valid 就会无情地返回 False
            ```
        * **第三步：格式化推平机 (Formatter) —— 强制统一工业标准**

            - 只要校验通过，我们就必须让全量数据输出成绝对一致的格式。在数据工程里，国际通用的电话号码绝对标准叫 E.164（必须以 + 开头，紧跟国家码，没有任何空格和连字符）。

            ```python
            # 强制转化为 E.164 机器标准格式
            clean_number = phonenumbers.format_number(parsed_obj, phonenumbers.PhoneNumberFormat.E164)

            print(clean_number)
            # 输出: '+861012345678'
            ```
* **🧬 Pandas 缝合术：打造工业级流水线**

    - 既然掌握了核心 API，现在要把它和 Pandas 结合。由于 phonenumbers 不是 Pandas 原生的 C 语言底层库，需要借助 `.apply()` 配合自定义函数，逐行对数据进行手术。

    - 标准模板：

    ```python
    import pandas as pd
    import numpy as np
    import phonenumbers

    def scrub_phone(phone_str, default_region="CN"):
        """
        工业级电话号码清洗微创手术刀
        """
        # 1. 拦截空值：本身就是 NaN 的，直接放行
        if pd.isna(phone_str):
            return np.nan
            
        try:
            # 2. 强转字符串并解析
            parsed_num = phonenumbers.parse(str(phone_str), default_region)
            
            # 3. 物理校验
            if phonenumbers.is_valid_number(parsed_num):
                # 4. 合法则输出 E.164 纯净格式
                return phonenumbers.format_number(parsed_num, phonenumbers.PhoneNumberFormat.E164)
            else:
                # 不合法（乱码/残缺），宣判为 NaN
                return np.nan
                
        except phonenumbers.NumberParseException:
            # 抓取解析崩溃（比如全是英文字母），直接转化为 NaN
            return np.nan

    # 模拟一批极其恶劣的靶场数据
    df = pd.DataFrame({
        'raw_phone': ['+86 138-1234-5678', '010 1234 5678', '9999', 'Disconnected', np.nan]
    })

    # 终极一刀：通过 apply 将函数拍在整列数据上
    df['clean_phone'] = df['raw_phone'].apply(scrub_phone)

    print(df)
    ```